# Module 03b — Vehicle mass

*What kind of car does each plant build, and how has that changed?*

Modules 01–03 asked **where** employment sits, **how** production is organised and **which** models each plant builds. This asks what those models **weigh** — the part of structural change a model-count cannot see. A plant can build the same nameplates for a decade while the cars themselves gain 200 kg.

Two movements are in play, and the point of the module is to keep them apart:

1. **Cars got bigger.** Combustion cars gained ~95 kg with no change of powertrain.
2. **Powertrains changed.** Battery cars are ~510 kg heavier, and went from nothing to an eighth of registrations.

Reported as one number they are indistinguishable, and a reader can attribute the whole rise to electrification. Split, they are two findings.

In [ ]:
import os
%load_ext autoreload
%autoreload 2

import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import foundation as fd
import module03_product as m3
import eea_crosswalk as xw
# --- archive bootstrap: resolve paths from the project root ---------------
# The original was CONFIG.data_dir = './data', which assumed the notebook ran
# beside a data/ folder. In the archived layout the inputs live in
# 02_data_inputs/, so the path is taken from specular_paths instead and this
# notebook works from wherever it is opened.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent))   # 01_code/
from specular_paths import P
fd.CONFIG.data_dir = str(P.inputs)
fd.CONFIG.out_dir  = str(P.outputs)
print('data:', fd.CONFIG.data_dir)
fd.CONFIG.out_dir  = './outputs'
print('data:', os.path.abspath(fd.CONFIG.data_dir))

## 1. The crosswalk — how model names were joined to EEA

EEA's commercial-name field is free text typed by manufacturers: one car appears as `A 180`, `A180`, `A 180 CDI`, `A 180 D 4MATIC`. Our side carries generation codes EEA never uses (`GOLF MK2`, `A3 8V`). An exact join reaches 48%.

`eea_crosswalk` normalises both sides to a model stem and matches on it, aggregating variants with registration weights — which is also the statistic we want: *the mass of the Golf* is the weighted mean over all Golf versions sold.

In [ ]:
mp = m3.model_panel()
cw = xw.build_crosswalk(mp)
rep = xw.match_report(mp, cw)

display(rep)
print(f"coverage {rep.attrs['coverage_pct']:.1f}%  "
      f"(precise {rep.attrs['precise_pct']:.1f}%, remainder coarse but valid)")

# sanity: do well-known models come out at plausible masses?
for b, m in [('Volkswagen','Golf'), ('Volkswagen','Polo'), ('Porsche','911'),
             ('Mercedes-Benz','S-Class'), ('Renault','Clio'), ('BMW','X5')]:
    r = cw[(cw.brand == b) & (cw.model == m)]
    if len(r):
        print(f'  {b:14s} {m:10s} {r.mass_kg.iloc[0]:7.0f} kg   ({r.rule.iloc[0]})')

## 2. Mass per plant

**The weighting problem, stated plainly.** We know which models a plant builds and what each weighs. We do *not* know how many of each it built — `production_value` is a plant total, and `production_volume_per_model` exists in the source schema as an empty column that was never populated.

So a plant average has to assume something about the mix inside it. Two assumptions bracket the answer and both are computed:

| | assumption |
|---|---|
| `equal` | every model the plant builds counts the same |
| `registrations` | models weighted by their European registrations |

Neither is right. The gap between them is the honest size of the uncertainty.

In [ ]:
pmm = m3.plant_model_mass()
pm  = m3.plant_mass(pmm, weighting='equal')

print(f'{pmm.plant_uid.nunique()} assembly plants, '
      f"{100*pmm.mass_kg.notna().mean():.0f}% of model rows carry a mass")
display(m3.coverage_report(pmm).tail(6).round(1))

fig = m3.fig_plant_mass(pm, year=2023, save=str(fd.CONFIG.out('fig03b_plant_mass.png'))); plt.show()

> The ordering is the Module 01 story told through the product. German premium plants — Munich, Emden, Leipzig, Sindelfingen, Neckarsulm, Dingolfing — sit at 1,900–2,200 kg; Spanish, French and Italian plants sit at 1,180–1,370 kg. The core keeps the large, high-value vehicles. That is **functional specialisation**, not simply relocation.

## 3. Balance the panel — before computing any trend

The raw production-weighted series rests on **5 plants in 2006, 22 in 2016 and 12 in 2025**. A trend over that mostly records which plants happened to publish an output figure. In Module 02 the same artefact manufactured two findings that did not survive balancing.

Choose the window on the evidence, not by convention.

In [ ]:
print('Requiring an observed production figure:')
display(m3.balance_report(pmm, 'production'))

print('Requiring only a model list:')
display(m3.balance_report(pmm, 'equal'))

> **Production weighting cannot be balanced.** Only 5–6 plants report an output figure in every year of any usable window, against 31 that report a model list over 2012–2023. So the headline series weights plants equally on a balanced panel, and the production-weighted series is shown only as a caution. This is a limit of the source, and it is stated rather than worked around.

## 3b. Mass per plant, across years

Section 2 is a single-year cross-section. This is the same measure followed **through time**, on the balanced panel — the view in which a plant switching product shows up.

> Ford Genk closed in December 2014 but its Wikipedia model list persists to 2025, and because those stale rows are *complete* they survive panel balancing. `plant_model_mass()` drops model-years recorded after a known closure by default; it prints a warning saying which.

In [ ]:
tr = m3.plant_trajectories(pmm, window=(2012, 2023))
fig = m3.fig_plant_trajectories(pmm, window=(2012, 2023),
                                 save=str(fd.CONFIG.out('fig03b_plant_trajectories.png'))); plt.show()

display(tr[['plant_name','country_iso2','mass_y0','mass_y1','change_kg','change_pct']].round(1))

> The large moves are **product switches, not drift**. Audi Brussels +924 kg (A1 hatchback → Q8 e-tron SUV), Zwickau +361 (Golf → ID.3/ID.4), Cologne +360 (Fiesta → electric SUV), Dresden −465 (Phaeton → ID.3). Twenty of thirty plants build heavier cars than in 2012, median +65 kg.

> Several plants sit at exactly 0: their recorded portfolio did not change over the window. That is a statement about the *source*, which records model presence from article snapshots, not proof that nothing changed on the line.

## 4. The European trend

In [ ]:
fig = m3.fig_mass_trend(window=(2012, 2023),
                         save=str(fd.CONFIG.out('fig03b_mass_trend.png'))); plt.show()

t = m3.mass_trend(pmm, 'equal', 'equal', window=(2012, 2023))
display(t[['year','mean_mass_kg','n_plants']].round(0))

## 5. Cars getting bigger, versus powertrains changing

Measured on EEA registrations directly rather than through our plant sample, because this is a statement about the European fleet.

The decomposition is **symmetric (Törnqvist)**: each class's mass change is weighted by its average share, each class's share change by its average mass. A naive version anchored at 2010 gives battery cars — which barely existed then — the same baseline as everything else, and the composition term collapses to zero by construction.

In [ ]:
dec = m3.powertrain_decomposition()
fig = m3.fig_powertrain(dec, save=str(fd.CONFIG.out('fig03b_powertrain.png'))); plt.show()

r = dec.iloc[-1]
print(f"{dec.attrs['y0']} -> {dec.attrs['y1']}:  total {r.total_change:+.0f} kg")
print(f"   cars of each type getting heavier : {r.within_effect:+6.0f} kg")
print(f"   shift to heavier powertrains      : {r.between_effect:+6.0f} kg")
print(f"   unexplained interaction           : {r.interaction:+6.0f} kg"
      f"  ({abs(100*r.interaction/r.total_change):.0f}% of total)")

## 6. How much does the unknown model mix matter?

In [ ]:
sens = m3.weighting_sensitivity(pmm)
fig = m3.fig_weighting_sensitivity(sens,
                                    save=str(fd.CONFIG.out('fig03b_sensitivity.png'))); plt.show()

print(f'median gap {sens.gap_pct.median():+.1f}%  |  '
      f'90th pct {sens.gap_pct.abs().quantile(.9):.1f}%  |  '
      f'max {sens.gap_pct.abs().max():.1f}%')

ex = m3.exact_mass_plants(pmm)
print(f'\n{len(ex)} plant-years built a single model, so their average mass '
      'needs no assumption at all.')
display(ex[['plant_name','year','model','mass_kg','production_value']].head(8))

> These single-model rows **cannot** validate the weighting choice — with one model both schemes give the same answer, so they carry no information about which is better. They are the rows where the *level* is certain, and that is all they are used for.

## 6b. A year-specific crosswalk

Everything above uses one mass per model for the whole period. That is defensible for mass — a Golf is roughly a Golf's weight throughout — but **wrong for CO2**, which fell by a quarter over the window and does not exist before 2018.

`build_crosswalk(by_year=True)` collapses the matched EEA rows *per year* instead, so a model carries its own trajectory.

In [ ]:
pmy = m3.plant_model_mass(by_year=True)

print(f'yearly panel: {len(pmy):,} rows | with mass {pmy.mass_kg.notna().sum():,}'
      f' | with CO2 {pmy.co2_g_km.notna().sum():,}')

cy = xw.build_crosswalk(mp, by_year=True)
display(cy[(cy.brand=='Volkswagen') & (cy.model=='Golf')]
        [['year','mass_kg','co2_g_km','bev_share']].tail(8).round(1))

## 6c. Tailpipe CO2 per plant

**Two limits, both structural.**

*The measure.* This is tailpipe CO2 under Reg. (EU) 2019/631, **not life-cycle**. A battery car is 0 g/km by regulatory definition — the emissions of building its battery and generating its electricity sit outside the measure entirely. A plant that electrifies falls to zero here while its real footprint does not.

*The window.* WLTP replaced NEDC across 2017–2019, so the field is empty before 2018. The aggregator previously divided a NaN-summed numerator by *total* registrations and returned a confident **0.0 g/km for 2010–2017** — "not measured" rendered as "zero emissions", which are opposites. It now carries a separate CO2 denominator and returns NaN. The series starts in 2019.

In [ ]:
co2 = m3.plant_co2(pmy)
fig = m3.fig_plant_co2(co2, year=2023, save=str(fd.CONFIG.out('fig03b_co2.png'))); plt.show()

display(co2.groupby('year').agg(plants=('plant_uid','nunique'),
                                mean_co2=('mean_co2_g_km','mean')).round(1))

flagged = co2[co2.all_electric_recorded]
print(f'{len(flagged)} plant-years read 0 g/km because every model in their '
      'RECORDED portfolio is electric:')
display(flagged[['plant_name','year','n_models_co2','n_models_listed']].head(8))

> ⚠ **Check these before quoting any single plant.** BMW Munich reads 0 g/km for 2023 because its recorded model list that year is four i4 variants — it plainly also built the 3-series. The model list comes from article snapshots and can miss combustion models still in production. The flag says *verify*, it does not delete the row.

## 6d. Did the plants that electrified also get heavier?

The fleet-level decomposition in section 5 said cars growing (+120 kg) mattered more than the powertrain shift (+61 kg). This asks the same question **across plants**.

In [ ]:
ev = m3.electrification_vs_mass(pmy, window=(2012, 2023))
fig = m3.fig_electrification_vs_mass(ev,
                                      save=str(fd.CONFIG.out('fig03b_evmass.png'))); plt.show()

print(f"{ev.attrs['n_plants']} plants | Pearson r = {ev.attrs['pearson_r']:.2f}"
      f" | Spearman rho = {ev.attrs['spearman_r']:.2f}")
display(ev[['plant_name','country_iso2','d_bev_pp','d_mass_kg']].round(1))

> **The association is weak — r = 0.10.** Plants that electrified more did gain more mass (median +233 kg against +102 kg for those that did not electrify at all), but the twelve plants with *no* change in BEV share still gained a median **+102 kg**. Mass rose where electrification did not happen, which is the plant-level echo of the fleet-level result.

> **Descriptive only.** Twenty plants, no controls, and both axes are computed from the same model list — a plant adding an electric SUV moves on both by construction. This is an association worth reporting and not an effect worth estimating.

## 7. Mass by bloc — the link back to Module 01

In [ ]:
bloc = m3.mass_by_bloc(pmm)
piv = bloc.pivot(index='year', columns='bloc', values='mean_mass_kg')
display(piv.loc[2012:2023].round(0))

print('plants per bloc in 2023:')
display(bloc[bloc.year == 2023][['bloc','n_plants']])

> **Read this one carefully.** The core rises from 1,564 to 1,655 kg while the southern periphery falls from 1,460 to 1,366 — consistent with the core holding larger vehicles. But the peripheral blocs rest on **four plants each**, so a single plant entering or leaving moves the line. Treat it as suggestive and say so in the text; the plant-level chart in section 2 carries the same finding on firmer ground.

## 8. Save

In [ ]:
cw.to_csv('outputs/tbl03b_eea_crosswalk.csv', index=False)
cy.to_csv('outputs/tbl03b_eea_crosswalk_yearly.csv', index=False)
pm.to_csv('outputs/tbl03b_plant_mass.csv', index=False)
co2.to_csv('outputs/tbl03b_plant_co2.csv', index=False)
ev.to_csv('outputs/tbl03b_electrification_vs_mass.csv', index=False)
dec.to_csv('outputs/tbl03b_powertrain_decomposition.csv', index=False)
sens.to_csv('outputs/tbl03b_weighting_sensitivity.csv', index=False)
print('saved to', os.path.abspath('outputs'))